<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260428.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
apt-get update -q
apt-get install -y -q mariadb-server
service mariadb start

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,863 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:14 http://archi

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
%%bash

mysql -u root <<'EOF'
CREATE DATABASE IF NOT EXISTS finance_lab CHARACTER SET utf8mb4 COLLATE utf8mb4_general_ci;
USE finance_lab;
CREATE USER IF NOT EXISTS 'testuser'@'localhost' IDENTIFIED BY '1234';
GRANT ALL PRIVILEGES ON finance_lab.* TO 'testuser'@'localhost';
FLUSH PRIVILEGES;

DROP TABLE IF EXISTS account_transactions;
DROP TABLE IF EXISTS accounts;
DROP TABLE IF EXISTS users;

CREATE TABLE users (
  id BIGINT PRIMARY KEY AUTO_INCREMENT,
  name VARCHAR(50) NOT NULL,
  email VARCHAR(100) NOT NULL UNIQUE,
  role ENUM('user', 'admin') NOT NULL DEFAULT 'user',
  status ENUM('active', 'inactive') NOT NULL DEFAULT 'active',
  created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE accounts (
  id BIGINT PRIMARY KEY AUTO_INCREMENT,
  user_id BIGINT NOT NULL,
  account_number VARCHAR(30) NOT NULL UNIQUE,
  account_name VARCHAR(100) NOT NULL,
  balance DECIMAL(15, 2) NOT NULL DEFAULT 0,
  status ENUM('active', 'closed') NOT NULL DEFAULT 'active',
  created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
  CONSTRAINT fk_accounts_users
    FOREIGN KEY (user_id) REFERENCES users(id)
);

CREATE TABLE account_transactions (
  id BIGINT PRIMARY KEY AUTO_INCREMENT,
  account_id BIGINT NOT NULL,
  transaction_type ENUM('deposit', 'withdraw', 'transfer') NOT NULL,
  amount DECIMAL(15, 2) NOT NULL,
  description VARCHAR(255),
  created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
  CONSTRAINT fk_transactions_accounts
    FOREIGN KEY (account_id) REFERENCES accounts(id)
);

INSERT INTO users (name, email, role, status) VALUES
('김민수', 'minsu@test.com', 'user', 'active'),
('이서연', 'seoyeon@test.com', 'user', 'active'),
('박관리', 'admin@test.com', 'admin', 'active');

INSERT INTO accounts (user_id, account_number, account_name, balance, status) VALUES
(1, '100-111-001', '생활비 계좌', 1500000, 'active'),
(1, '100-111-002', '저축 계좌', 5000000, 'active'),
(2, '100-222-001', '월급 계좌', 3200000, 'active');

INSERT INTO account_transactions
(account_id, transaction_type, amount, description, created_at)
VALUES
(1, 'deposit', 1000000, '급여 입금', '2026-04-01 09:00:00'),
(1, 'withdraw', 50000, '카페 결제', '2026-04-02 12:10:00'),
(1, 'withdraw', 120000, '마트 결제', '2026-04-03 18:20:00'),
(2, 'deposit', 3000000, '저축 입금', '2026-04-04 10:00:00'),
(3, 'deposit', 3200000, '급여 입금', '2026-04-01 09:30:00'),
(3, 'transfer', 500000, '타행 이체', '2026-04-05 14:00:00');

SELECT 'setup complete' AS result;
EOF

result
setup complete




---



In [5]:
%%bash

mysql -u root finance_lab <<'EOF'
SELECT
  u.id AS user_id,
  u.name,
  u.email,
  a.account_number,
  a.account_name,
  a.balance
FROM users u
INNER JOIN accounts a
  ON u.id = a.user_id
ORDER BY u.id, a.id;
EOF

user_id	name	email	account_number	account_name	balance
1	김민수	minsu@test.com	100-111-001	생활비 계좌	1500000.00
1	김민수	minsu@test.com	100-111-002	저축 계좌	5000000.00
2	이서연	seoyeon@test.com	100-222-001	월급 계좌	3200000.00


In [7]:
%%bash

mysql -u root finance_lab <<'EOF'
SELECT
  u.id,
  u.name,
  u.email,
  a.account_number,
  a.balance
FROM users u
LEFT JOIN accounts a
  ON u.id = a.user_id
ORDER BY u.id;
EOF

id	name	email	account_number	balance
1	김민수	minsu@test.com	100-111-001	1500000.00
1	김민수	minsu@test.com	100-111-002	5000000.00
2	이서연	seoyeon@test.com	100-222-001	3200000.00
3	박관리	admin@test.com	NULL	NULL


In [8]:
%%bash

mysql -u root finance_lab <<'EOF'
SELECT
  a.id AS account_id,
  a.account_number,
  COUNT(t.id) AS transaction_count,
  SUM(t.amount) AS total_amount,
  AVG(t.amount) AS average_amount,
  MAX(t.amount) AS max_amount
FROM accounts a
LEFT JOIN account_transactions t
  ON a.id = t.account_id
GROUP BY a.id, a.account_number
ORDER BY a.id;
EOF

account_id	account_number	transaction_count	total_amount	average_amount	max_amount
1	100-111-001	3	1170000.00	390000.000000	1000000.00
2	100-111-002	1	3000000.00	3000000.000000	3000000.00
3	100-222-001	2	3700000.00	1850000.000000	3200000.00


In [9]:
%%bash

mysql -u root finance_lab <<'EOF'
SELECT
  a.account_number,
  COUNT(t.id) AS transaction_count,
  SUM(t.amount) AS total_amount
FROM accounts a
JOIN account_transactions t
  ON a.id = t.account_id
GROUP BY a.account_number
HAVING SUM(t.amount) >= 1000000;
EOF

account_number	transaction_count	total_amount
100-111-001	3	1170000.00
100-111-002	1	3000000.00
100-222-001	2	3700000.00


In [10]:
%%bash

mysql -u root finance_lab <<'EOF'
SELECT
  a.account_number,
  SUM(CASE WHEN t.transaction_type = 'deposit' THEN t.amount ELSE 0 END) AS deposit_total,
  SUM(CASE WHEN t.transaction_type = 'withdraw' THEN t.amount ELSE 0 END) AS withdraw_total,
  SUM(CASE WHEN t.transaction_type = 'transfer' THEN t.amount ELSE 0 END) AS transfer_total
FROM accounts a
LEFT JOIN account_transactions t
  ON a.id = t.account_id
GROUP BY a.account_number;
EOF

account_number	deposit_total	withdraw_total	transfer_total
100-111-001	1000000.00	170000.00	0.00
100-111-002	3000000.00	0.00	0.00
100-222-001	3200000.00	0.00	500000.00


In [11]:
%%bash

mysql -u root finance_lab <<'EOF'
SELECT
  account_number,
  account_name,
  balance
FROM accounts
WHERE balance > (
  SELECT AVG(balance)
  FROM accounts
);
EOF

account_number	account_name	balance
100-111-002	저축 계좌	5000000.00


In [12]:
%%bash

mysql -u root finance_lab <<'EOF'
SELECT
  a.id,
  a.account_number,
  a.account_name
FROM accounts a
WHERE EXISTS (
  SELECT 1
  FROM account_transactions t
  WHERE t.account_id = a.id
);
EOF

id	account_number	account_name
1	100-111-001	생활비 계좌
2	100-111-002	저축 계좌
3	100-222-001	월급 계좌


In [13]:
%%bash

mysql -u root finance_lab <<'EOF'
WITH user_transaction_summary AS (
  SELECT
    u.id AS user_id,
    u.name,
    SUM(t.amount) AS total_amount
  FROM users u
  JOIN accounts a
    ON u.id = a.user_id
  JOIN account_transactions t
    ON a.id = t.account_id
  GROUP BY u.id, u.name
)
SELECT
  user_id,
  name,
  total_amount
FROM user_transaction_summary
WHERE total_amount >= 1000000
ORDER BY total_amount DESC;
EOF

user_id	name	total_amount
1	김민수	4170000.00
2	이서연	3700000.00


In [14]:
%%bash

mysql -u root finance_lab <<'EOF'
SELECT
  t.account_id,
  t.transaction_type,
  t.amount,
  t.created_at,
  ROW_NUMBER() OVER (
    PARTITION BY t.account_id
    ORDER BY t.created_at DESC
  ) AS recent_rank
FROM account_transactions t;
EOF

account_id	transaction_type	amount	created_at	recent_rank
1	withdraw	120000.00	2026-04-03 18:20:00	1
1	withdraw	50000.00	2026-04-02 12:10:00	2
1	deposit	1000000.00	2026-04-01 09:00:00	3
2	deposit	3000000.00	2026-04-04 10:00:00	1
3	transfer	500000.00	2026-04-05 14:00:00	1
3	deposit	3200000.00	2026-04-01 09:30:00	2


In [15]:
%%bash

mysql -u root finance_lab <<'EOF'
WITH ranked_transactions AS (
  SELECT
    t.*,
    ROW_NUMBER() OVER (
      PARTITION BY t.account_id
      ORDER BY t.created_at DESC
    ) AS rn
  FROM account_transactions t
)
SELECT
  account_id,
  transaction_type,
  amount,
  description,
  created_at
FROM ranked_transactions
WHERE rn = 1;
EOF

account_id	transaction_type	amount	description	created_at
1	withdraw	120000.00	마트 결제	2026-04-03 18:20:00
2	deposit	3000000.00	저축 입금	2026-04-04 10:00:00
3	transfer	500000.00	타행 이체	2026-04-05 14:00:00
